In [1]:
import pandas as pd

# ==========================================
# 1. LOAD KETIGA SUMBER DATASET (7 File CSV)
# ==========================================

# Sumber 1: Hasil Data Crawling
df_crawl_rating = pd.read_csv('Hasil Data Crawling/tabel_rating.csv')
df_crawl_wisata = pd.read_csv('Hasil Data Crawling/tabel_wisata.csv')

# Sumber 2: Indonesia Tourism Destination by A_Prabowo
df_prabowo_pkg = pd.read_csv('Indonesia Tourism Destination by A_Prabowo/package_tourism.csv')
df_prabowo_rating = pd.read_csv('Indonesia Tourism Destination by A_Prabowo/tourism_rating.csv')
df_prabowo_tourism = pd.read_csv('Indonesia Tourism Destination by A_Prabowo/tourism_with_id.csv')
df_prabowo_user = pd.read_csv('Indonesia Tourism Destination by A_Prabowo/user.csv')

# Sumber 3: Tourism Data on Java by masdarul rizqi
df_java = pd.read_csv('Tourism Data on Java by masdarul rizqi/Java.csv')

# ==========================================
# 2. FUNGSI UNTUK CEK INFO & SHAPE DATASET
# ==========================================

def cek_info_dataset(nama, df):
    print(f"\n{'='*50}")
    print(f"📊 DATASET: {nama}")
    print(f"{'='*50}")
    print(f"▶ 1. Total Baris dan Kolom (Shape): {df.shape}")
    print(f"▶ 2. Daftar Nama Kolom:\n{list(df.columns)}\n")
    print("▶ 3. Struktur Data (Info):")
    df.info()
    print("\n")

# Menjalankan fungsi untuk melihat info masing-masing tabel (Fokus ke tabel utama tempat wisata & rating)
cek_info_dataset("Crawling - Wisata", df_crawl_wisata)
cek_info_dataset("Prabowo - Tourism with ID", df_prabowo_tourism)
cek_info_dataset("Masdarul Rizqi - Java", df_java)

# ==========================================
# 3. CEK KETERSEDIAAN DATA KOTA JOGJA & SOLO
# ==========================================
print(f"{'='*50}\n🔍 PENGECEKAN KOTA JOGJA & SOLO\n{'='*50}")

def cek_kota(nama, df):
    # Mencari tahu nama kolom yang merepresentasikan kota/lokasi
    col_kota = None
    for col in df.columns:
        if col.lower() in ['city', 'kota', 'city_name', 'location']:
            col_kota = col
            break
            
    if col_kota:
        print(f"\n📍 Distribusi Kota di dataset '{nama}' (Berdasarkan kolom '{col_kota}'):")
        # Filter spesifik kata Jogja, Yogyakarta, Solo, Surakarta
        mask = df[col_kota].astype(str).str.contains('Jogja|Yogyakarta|Solo|Surakarta', case=False, na=False)
        hasil_filter = df[mask][col_kota].value_counts()
        
        if hasil_filter.empty:
            print("   -> Tidak ditemukan data eksplisit Jogja/Solo di kolom ini.")
        else:
            print(hasil_filter.to_string())
    else:
        print(f"\n📍 Dataset '{nama}': Tidak ada kolom yang bernama 'City', 'Kota', atau 'Location'.")

# Cek di ketiga dataset utama wisata
cek_kota("Crawling - Wisata", df_crawl_wisata)
cek_kota("Prabowo - Tourism with ID", df_prabowo_tourism)
cek_kota("Masdarul Rizqi - Java", df_java)


📊 DATASET: Crawling - Wisata
▶ 1. Total Baris dan Kolom (Shape): (197, 5)
▶ 2. Daftar Nama Kolom:
['ID_Wisata', 'Nama_Wisata', 'Kota', 'Kategori', 'Deskripsi_Wisata']

▶ 3. Struktur Data (Info):
<class 'pandas.DataFrame'>
RangeIndex: 197 entries, 0 to 196
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   ID_Wisata         197 non-null    str  
 1   Nama_Wisata       197 non-null    str  
 2   Kota              197 non-null    str  
 3   Kategori          197 non-null    str  
 4   Deskripsi_Wisata  197 non-null    str  
dtypes: str(5)
memory usage: 7.8 KB



📊 DATASET: Prabowo - Tourism with ID
▶ 1. Total Baris dan Kolom (Shape): (437, 13)
▶ 2. Daftar Nama Kolom:
['Place_Id', 'Place_Name', 'Description', 'Category', 'City', 'Price', 'Rating', 'Time_Minutes', 'Coordinate', 'Lat', 'Long', 'Unnamed: 11', 'Unnamed: 12']

▶ 3. Struktur Data (Info):
<class 'pandas.DataFrame'>
RangeIndex: 437 entries, 0 to 436
Data 

In [2]:
# ==========================================
# TAHAP 1: STANDARISASI DATASET 2 (PRABOWO)
# ==========================================

print("⏳ Memulai Tahap 1: Standarisasi Dataset Prabowo...")

# 1. Filtering Lokasi (Hanya Yogyakarta)
# Menggunakan .copy() agar tidak mengganggu dataset aslinya (mencegah SettingWithCopyWarning)
df_prabowo_clean = df_prabowo_tourism[df_prabowo_tourism['City'] == 'Yogyakarta'].copy()

# 2. Drop Kolom Sampah
kolom_dihapus = ['Time_Minutes', 'Coordinate', 'Lat', 'Long', 'Price', 'Unnamed: 11', 'Unnamed: 12']
df_prabowo_clean.drop(columns=kolom_dihapus, inplace=True)

# 3. Rename Kolom
perubahan_nama = {
    'Place_Id': 'ID_Wisata',
    'Place_Name': 'Nama_Wisata',
    'City': 'Kota',
    'Category': 'Kategori',
    'Description': 'Deskripsi_Wisata'
}
df_prabowo_clean.rename(columns=perubahan_nama, inplace=True)

# (Opsional tapi Best Practice) Menyamakan urutan kolom persis seperti Dataset 1
urutan_kolom = ['ID_Wisata', 'Nama_Wisata', 'Kota', 'Kategori', 'Deskripsi_Wisata']
df_prabowo_clean = df_prabowo_clean[urutan_kolom]

print("✅ Tahap 1 Selesai! Shape Dataset Prabowo sekarang:", df_prabowo_clean.shape)

# ==========================================
# TAHAP 2: PENGGABUNGAN (CONCATENATION)
# ==========================================

print("\n⏳ Memulai Tahap 2: Menggabungkan Dataset 1 dan Dataset 2...")

# Menggabungkan secara vertikal (baris bertambah ke bawah)
# ignore_index=True memastikan index direset dari 0 berurutan
df_gabungan = pd.concat([df_crawl_wisata, df_prabowo_clean], ignore_index=True)

print(f"✅ Tahap 2 Selesai! Total baris sementara setelah digabung: {df_gabungan.shape[0]} baris")

# ==========================================
# TAHAP 3: DEDUPLIKASI (HANDLING DUPLICATES)
# ==========================================

print("\n⏳ Memulai Tahap 3: Deduplikasi Tempat Wisata...")

# BEST PRACTICE: Sebelum menghapus duplikat, kita seragamkan dulu teksnya (huruf kecil semua & hilangkan spasi sisa di ujung)
# Ini penting karena "Candi Prambanan" dan "candi prambanan " dianggap dua data berbeda oleh sistem jika tidak dirapikan.
df_gabungan['Nama_Normalisasi'] = df_gabungan['Nama_Wisata'].str.lower().str.strip()

# Menghapus duplikat berdasarkan kolom nama yang sudah dinormalisasi
# keep='first' berarti kita menyimpan kemunculan pertama dan menghapus sisanya
df_gabungan.drop_duplicates(subset=['Nama_Normalisasi'], keep='first', inplace=True)

# Membuang kolom bantuan 'Nama_Normalisasi' karena sudah tidak dipakai
df_gabungan.drop(columns=['Nama_Normalisasi'], inplace=True)

# Reset index kembali agar rapi setelah ada baris yang terhapus
df_gabungan.reset_index(drop=True, inplace=True)

print("✅ Tahap 3 Selesai!")
print(f"🎯 TOTAL BARIS DATA FINAL BERSIH: {df_gabungan.shape[0]} baris")

# Melihat 5 data teratas untuk memastikan hasilnya rapi
print("\n🔍 Cuplikan 5 Data Pertama:")
display(df_gabungan.head())

⏳ Memulai Tahap 1: Standarisasi Dataset Prabowo...
✅ Tahap 1 Selesai! Shape Dataset Prabowo sekarang: (126, 5)

⏳ Memulai Tahap 2: Menggabungkan Dataset 1 dan Dataset 2...
✅ Tahap 2 Selesai! Total baris sementara setelah digabung: 323 baris

⏳ Memulai Tahap 3: Deduplikasi Tempat Wisata...
✅ Tahap 3 Selesai!
🎯 TOTAL BARIS DATA FINAL BERSIH: 276 baris

🔍 Cuplikan 5 Data Pertama:


,ID_Wisata,Nama_Wisata,Kota,Kategori,Deskripsi_Wisata
0,W_73ace55b,Jl. Malioboro,Yogyakarta,Budaya/Sejarah,Destinasi wisata Jl. Malioboro berlokasi di Yo...
1,W_6f17a023,Keraton Ngayogyakarta Hadiningrat,Yogyakarta,Budaya/Sejarah,"Dibangun pada abad ke-18, istana megah ini mas..."
2,W_95a7288c,Kampung Wisata Taman Sari,Yogyakarta,Budaya/Sejarah,Bekas taman kerajaan dari abad ke-18 dengan ko...
3,W_05ed3f8c,Tugu Jogja,Yogyakarta,Budaya/Sejarah,Tugu putih & emas bersejarah di penyeberangan ...
4,W_7f931485,Museum Benteng Vredeburg,Yogyakarta,Budaya/Sejarah,Museum yang dulunya benteng kolonial ini menam...


In [3]:
# ==========================================
# TAHAP 4: MENYIMPAN DATASET WISATA BERSIH
# ==========================================

print("⏳ Menyimpan Dataset Wisata Bersih...")
# Menyimpan ke format CSV tanpa menyertakan index bawaan Pandas
df_gabungan.to_csv('Data_Wisata_Clean.csv', index=False)
print("✅ Tahap 4 Selesai! File 'Data_Wisata_Clean.csv' berhasil dibuat di foldermu.\n")


# ==========================================
# TAHAP 5: MENYELARASKAN & MENGGABUNGKAN RATING
# ==========================================

print("⏳ Memulai Tahap 5: Sinkronisasi Dataset Rating...")

# 1. Menyiapkan Data Rating Crawling
df_rating_crawl = df_crawl_rating.copy()
# Menyeragamkan nama kolom (asumsi kolom berisi User_ID, ID_Wisata, Rating)
df_rating_crawl.columns = ['User_ID', 'ID_Wisata', 'Rating'] 

# 2. Menyiapkan Data Rating Prabowo
df_rating_prabowo = df_prabowo_rating.copy()
# Rename sementara untuk konsistensi
df_rating_prabowo.rename(columns={'User_Id': 'User_ID', 'Place_Ratings': 'Rating'}, inplace=True)

# TANTANGAN: Kita butuh 'ID_Wisata' (string) yang baru, bukan 'Place_Id' (angka).
# Solusi: Join dengan tabel tourism asli untuk dapat nama, lalu join dengan data bersih kita.

# a. Gabung dengan tourism Prabowo untuk ambil 'Place_Name'
df_rating_prabowo = df_rating_prabowo.merge(df_prabowo_tourism[['Place_Id', 'Place_Name']], on='Place_Id', how='left')

# b. Buat kolom normalisasi nama untuk pencocokan
df_rating_prabowo['Nama_Normalisasi'] = df_rating_prabowo['Place_Name'].str.lower().str.strip()
df_gabungan['Nama_Normalisasi'] = df_gabungan['Nama_Wisata'].str.lower().str.strip()

# c. Ambil 'ID_Wisata' yang sah dari df_gabungan berdasarkan nama yang cocok (inner join otomatis membuang rating untuk wisata luar Jogja)
df_rating_prabowo_mapped = df_rating_prabowo.merge(df_gabungan[['Nama_Normalisasi', 'ID_Wisata']], on='Nama_Normalisasi', how='inner')

# d. Ambil hanya 3 kolom utama yang kita butuhkan
df_rating_prabowo_final = df_rating_prabowo_mapped[['User_ID', 'ID_Wisata', 'Rating']]

# 3. Gabungkan Kedua Dataset Rating
df_rating_gabungan = pd.concat([df_rating_crawl, df_rating_prabowo_final], ignore_index=True)

# 4. Deduplikasi Rating (Jaga-jaga jika 1 user memberi rating ke 1 tempat yang sama lebih dari sekali)
df_rating_gabungan.drop_duplicates(subset=['User_ID', 'ID_Wisata'], keep='last', inplace=True)

# 5. Simpan Dataset Rating Bersih
df_rating_gabungan.to_csv('Data_Rating_Clean.csv', index=False)

print("✅ Tahap 5 Selesai! Mapping ID berhasil dilakukan tanpa error.")
print(f"🎯 TOTAL BARIS RATING BERSIH: {df_rating_gabungan.shape[0]} baris")

# Menghapus kolom bantuan normalisasi di df_gabungan agar tetap rapi
df_gabungan.drop(columns=['Nama_Normalisasi'], inplace=True)

print("\n🔍 Cuplikan 5 Data Rating Pertama:")
display(df_rating_gabungan.head())

⏳ Menyimpan Dataset Wisata Bersih...
✅ Tahap 4 Selesai! File 'Data_Wisata_Clean.csv' berhasil dibuat di foldermu.

⏳ Memulai Tahap 5: Sinkronisasi Dataset Rating...
✅ Tahap 5 Selesai! Mapping ID berhasil dilakukan tanpa error.
🎯 TOTAL BARIS RATING BERSIH: 49457 baris

🔍 Cuplikan 5 Data Rating Pertama:


,User_ID,ID_Wisata,Rating
0,U_396307,W_73ace55b,5
1,U_844830,W_73ace55b,5
2,U_199948,W_73ace55b,5
3,U_686252,W_73ace55b,5
4,U_796278,W_73ace55b,5
